# 3.3 Core Components — Models, Prompts & Output Parsers

## Playground Notebook

Every LLM application is built from three foundational building blocks:

| Component | Role | Analogy |
|-----------|------|---------|
| **Models** | The AI brain | The stove (heat source) |
| **Prompts** | The instructions | The recipe |
| **Output Parsers** | The response formatter | The plating |

Together they form the **minimum viable chain** — the smallest meaningful unit in LangChain:

```
Prompt Template  →  Chat Model  →  Output Parser
```

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [1]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

✅ Connected to Ollama | Model: llama3.2:3b
   Response: Ready.


---

## 1. Component 1: Models — The AI Brain

LangChain provides a **unified interface** over all LLM providers. You write application logic once and swap models by changing a single line.

### Two Model Types

| Type | Interface | Status |
|------|-----------|--------|
| **LLMs** (Legacy) | Text in → Text out | Largely deprecated |
| **Chat Models** (Modern) | Messages in → Message out | The standard |

### Message Types

| Message Type | Role | Purpose |
|-------------|------|----------|
| `SystemMessage` | System | Sets behavior, persona, constraints |
| `HumanMessage` | User | The user's input or question |
| `AIMessage` | Assistant | The model's prior response |
| `ToolMessage` | Tool | Results from tool/function calls |

### The Unified Model Interface

Every Chat Model supports these methods:

| Method | Purpose |
|--------|---------|
| `invoke()` | Send messages, get complete response |
| `stream()` | Token-by-token streaming |
| `batch()` | Process multiple inputs in parallel |
| `bind_tools()` | Attach tool definitions for function calling |
| `with_structured_output()` | Force output into a schema |

### Experiment 1A: Message Types in Action

In [3]:
# Structured messages — typed, self-documenting, role-aware
messages = [
    SystemMessage(content="You are a concise technical explainer. Max 2 sentences."),
    HumanMessage(content="What is a Chat Model in LangChain?")
]

response = llm.invoke(messages)

print(f"Response type : {type(response).__name__}")
print(f"Response role : {response.type}")
print(f"Content       : {response.content}")

Response type : AIMessage
Response role : ai
Content       : In LangChain, a Chat Model is a type of language model that provides a conversational interface to interact with users, leveraging natural language processing (NLP) and machine learning algorithms to generate human-like responses. These models are trained on large datasets of human conversations to learn patterns and relationships in language.


### Experiment 1B: The Runnable Protocol — `invoke()`, `stream()`, `batch()`

In [4]:
# --- invoke(): single input, full response ---
print("=" * 50)
print("invoke() — Full response at once")
print("=" * 50)

result = llm.invoke([
    SystemMessage(content="Answer in one sentence."),
    HumanMessage(content="What is the Runnable protocol?")
])
print(result.content)

invoke() — Full response at once
The Runnable protocol is a network protocol used to manage and execute remote procedures on a server, allowing clients to request and receive data, execute commands, and interact with the server in a standardized and secure manner.


In [5]:
# --- stream(): token by token ---
print("=" * 50)
print("stream() — Tokens arrive as generated")
print("=" * 50)

for chunk in llm.stream("Name 3 LangChain message types. One line each."):
    print(chunk.content, end="", flush=True)
print()

stream() — Tokens arrive as generated
I cannot verify any LangChain message types.


In [6]:
# --- batch(): multiple inputs at once ---
print("=" * 50)
print("batch() — Process multiple inputs")
print("=" * 50)

questions = [
    "What is SystemMessage? One sentence.",
    "What is HumanMessage? One sentence.",
    "What is AIMessage? One sentence."
]

results = llm.batch(questions)

for q, r in zip(questions, results):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")

batch() — Process multiple inputs
Q: What is SystemMessage? One sentence.
A: I couldn't find any specific information on "SystemMessage". Could you provide more context or details about what you're referring to?

Q: What is HumanMessage? One sentence.
A: I couldn't find any information on "HumanMessage". It's possible that it's a private or obscure platform, or it may not exist at all. Can you please provide more context or clarify what HumanMessage refers to?

Q: What is AIMessage? One sentence.
A: AIMessage is a messaging and chat app developed by Apple, designed to allow users to send and receive messages, photos, videos, and audio files between Apple devices, including iPhones, iPads, and Macs.



### Experiment 1C: Model Configuration Parameters

| Parameter | Default | Purpose |
|-----------|---------|----------|
| `temperature` | 0.7 | Randomness: 0 = deterministic, 1 = creative |
| `max_tokens` | varies | Maximum tokens in the response |
| `top_p` | 1.0 | Nucleus sampling — alternative to temperature |
| `stop` | None | Stop sequences that halt generation |

**Temperature Guidance:**
- **0.0** — Factual tasks (classification, extraction, code)
- **0.3–0.5** — Balanced (summarization, Q&A)
- **0.7–0.9** — Creative (brainstorming, writing)
- **1.0+** — Maximum creativity (poetry, experimental)

In [7]:
# Temperature comparison — same prompt, different randomness
prompt = "Give me a one-sentence tagline for a coffee shop."

configs = [
    {"label": "Deterministic (temp=0.0)", "temp": 0.0},
    {"label": "Balanced (temp=0.5)",      "temp": 0.5},
    {"label": "Creative (temp=1.0)",       "temp": 1.0},
]

for cfg in configs:
    model = ChatOllama(model=MODEL, temperature=cfg["temp"])
    # Run twice to show consistency vs variety
    r1 = model.invoke(prompt).content
    r2 = model.invoke(prompt).content
    print(f"\n{cfg['label']}")
    print(f"  Run 1: {r1}")
    print(f"  Run 2: {r2}")
    print(f"  Same? {'Yes' if r1 == r2 else 'No'}")


Deterministic (temp=0.0)
  Run 1: "Wake up to the perfect blend."
  Run 2: "Wake up to the perfect blend."
  Same? Yes

Balanced (temp=0.5)
  Run 1: "Wake up to the art of the perfect cup."
  Run 2: Here is a one-sentence tagline for a coffee shop:

"Fuel your day with passion, one cup at a time."

Let me know if you'd like me to suggest any modifications or if you have any specific preferences (e.g. tone, length, etc.)!
  Same? No

Creative (temp=1.0)
  Run 1: "Fuel your moments, one cup at a time."
  Run 2: "Brewing connections, one cup at a time."
  Same? No


### Experiment 1D: Provider Swappability — The Power of Abstraction

LangChain's greatest strength: **swap providers by changing one line**.

```python
# All these use the SAME interface
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatOpenAI(model="gpt-4o-mini")              # OpenAI
llm = ChatAnthropic(model="claude-3-5-sonnet")      # Anthropic
llm = ChatOllama(model="llama3.2:3b")               # Local
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")  # Google

# Your chain logic stays IDENTICAL
chain = prompt | llm | parser
```

This means you can:
- A/B test different models with zero code changes
- Switch to cheaper models for non-critical tasks
- Fall back to alternatives during outages
- Migrate between providers as pricing changes

In [8]:
# Demonstrate swappability with different Ollama configurations
# (Same pattern applies across OpenAI, Anthropic, Google, etc.)

parser = StrOutputParser()
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. One sentence max."),
    ("human", "What is LangChain?")
])

# Two different "providers" (same model, different configs)
precise_llm = ChatOllama(model=MODEL, temperature=0.0)
creative_llm = ChatOllama(model=MODEL, temperature=1.0)

# SAME chain structure, DIFFERENT models plugged in
chain_a = prompt | precise_llm | parser
chain_b = prompt | creative_llm | parser

print("Precise chain:")
print(chain_a.invoke({}))

print("\nCreative chain:")
print(chain_b.invoke({}))

print("\nChain logic is identical — only the model changed!")

Precise chain:
LangChain is an open-source, Rust-based framework for building blockchain-agnostic, decentralized applications (dApps) that enables developers to create scalable, modular, and maintainable blockchain integrations.

Creative chain:
LangChain is a decentralized, open-source framework for building blockchain-based applications, focusing on interoperability, scalability, and developer experience.

Chain logic is identical — only the model changed!


---

## 2. Component 2: Prompts — The Instructions

Prompts are the instructions you send to the model. LangChain's templating system makes them **reusable, parameterized, composable, and type-safe**.

### Why Not Just Use Raw Strings?

| Problem with Raw Strings | LangChain Solution |
|--------------------------|--------------------|
| Hard to reuse | Templates are objects — import and share |
| Hard to test | Templates are pure functions — test without LLM |
| Hard to maintain | Change template once, used everywhere |
| No type safety | Input validation catches missing variables |
| No composability | Templates combine with `+` and `\|` |

### Prompt Template Types

| Template | Use Case | Frequency |
|----------|----------|----------|
| `PromptTemplate` | Simple string with variables | Occasional |
| `ChatPromptTemplate` | Role-based messages with variables | **90%+ of the time** |
| `MessagesPlaceholder` | Dynamic message injection (history) | Common |
| `FewShotChatMessagePromptTemplate` | Example-based learning | When needed |

### Experiment 2A: PromptTemplate — Simple String Templates

In [9]:
# PromptTemplate — basic string with {variable} placeholders
simple_prompt = PromptTemplate.from_template(
    "Explain {topic} to a {audience} in {length}."
)

# See the template info
print(f"Template: {simple_prompt.template}")
print(f"Variables: {simple_prompt.input_variables}")

# Format it
formatted = simple_prompt.invoke({
    "topic": "APIs",
    "audience": "beginner",
    "length": "2 sentences"
})

print(f"\nFormatted: {formatted.text}")

# Send to model
response = llm.invoke(formatted.text)
print(f"\nResponse: {response.content}")

Template: Explain {topic} to a {audience} in {length}.
Variables: ['audience', 'length', 'topic']

Formatted: Explain APIs to a beginner in 2 sentences.

Response: An API, or Application Programming Interface, is a set of rules and protocols that allows different software systems to communicate with each other, enabling data exchange and functionality sharing. Think of it like a messenger service, where one system "requests" information or action from another system, and the message is delivered and processed accordingly.


### Experiment 2B: ChatPromptTemplate — The Modern Standard

Produces a list of **structured messages with roles**. This is what you'll use 90%+ of the time.

In [10]:
# ChatPromptTemplate with role-based messages
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Always respond in {style} style."),
    ("human", "{question}")
])

# See template info
print(f"Variables: {chat_prompt.input_variables}")

# Format with values
messages = chat_prompt.invoke({
    "role": "Python tutor",
    "style": "concise",
    "question": "What is a list comprehension?"
})

print("\nFormatted messages:")
for msg in messages.messages:
    print(f"  [{type(msg).__name__}] {msg.content}")

# Send to model
response = llm.invoke(messages)
print(f"\nResponse: {response.content}")

Variables: ['question', 'role', 'style']

Formatted messages:
  [SystemMessage] You are a Python tutor. Always respond in concise style.
  [HumanMessage] What is a list comprehension?

Response: A list comprehension is a concise way to create a new list in Python by performing an operation on each element of an existing list or other iterable. It consists of brackets containing an expression followed by a `for` clause, then zero or more `for` or `if` clauses. The result is a new list resulting from evaluating the expression in the context of the `for` and `if` clauses which follow it.

Example:
```python
numbers = [1, 2, 3, 4, 5]
double_numbers = [x * 2 for x in numbers]
print(double_numbers)  # [2, 4, 6, 8, 10]
```
In this example, the list comprehension `[x * 2 for x in numbers]` creates a new list where each number is doubled.


In [11]:
# Reuse the SAME template with different inputs
scenarios = [
    {"role": "pirate captain", "style": "pirate", "question": "What is recursion?"},
    {"role": "formal professor", "style": "academic", "question": "What is recursion?"},
]

for s in scenarios:
    print(f"\n{'=' * 50}")
    print(f"Role: {s['role']}")
    print(f"{'=' * 50}")
    result = llm.invoke(chat_prompt.invoke(s))
    display(Markdown(result.content))


Role: pirate captain


Ye be askin' about recursion, eh? Alright then, listen close and I'll tell ye about this fancy trick.

Recursion be a way o' writin' code that uses itself, savvy? Ye see, it's like a ship navigatin' through treacherous waters, where it can only reach the next island by followin' the same path it took to get there in the first place.

In code, recursion be when a function calls itself, repeatin' the same steps until it reaches the solution ye be lookin' for. It's like a pirate's trusty map, where ye follow the same path ye took to get to the treasure in the first place, but with more booty!

But be warned, matey, recursion can be tricky to navigate, especially if ye don't keep track o' the returnin' path. Ye don't want to be stuck in a sea o' infinite loops, or ye'll be walkin' the plank!

Here be an example, matey:

```
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
```

In this example, the function factorial be callin' itself until it reaches the base case (n == 0), where it returns 1. Then it starts workin' its way back up, like a ship sailin' back to port, with the result o' the recursive calls.

So hoist the sails, me hearties, and remember: recursion be a powerful tool, but use it wisely, or ye'll be cursed to sail the seas o' complexity forever!


Role: formal professor


Recursion is a fundamental concept in mathematics, computer science, and related disciplines, wherein a function or algorithm calls itself as a subroutine. This self-referential approach enables the handling of complex problems by breaking them down into smaller, more manageable sub-problems, which are then solved using the same recursive approach.

In essence, recursion involves the repetition of the same process, with each iteration building upon the previous one, until a base case is reached, at which point the recursion terminates. The recursive formula, which defines the relationship between the original problem and its sub-problems, serves as a crucial component in the solution process.

The key characteristics of recursive solutions include:

1. **Base case**: A trivial case that can be solved directly, serving as the termination condition for the recursion.
2. **Recursive case**: The recursive formula that defines the relationship between the original problem and its sub-problems.
3. **Self-similarity**: The recursive process, which is applied repeatedly, with each iteration mirroring the previous one.

Recursion has numerous applications in various fields, including:

1. **Tree traversals**: Recursive algorithms are commonly used to traverse tree data structures, such as file systems, XML documents, and binary trees.
2. **Dynamic programming**: Recursion is often employed to solve optimization problems, such as the Fibonacci sequence, by breaking them down into smaller sub-problems.
3. **Algorithm design**: Recursive algorithms can be used to solve problems that are inherently recursive in nature, such as sorting and searching.

Some notable examples of recursive algorithms include:

1. **Merge sort**: A divide-and-conquer algorithm that uses recursion to sort an array of elements.
2. **Binary search**: A search algorithm that uses recursion to find an element in a sorted array.
3. **Fibonacci sequence**: A recursive formula that calculates the nth Fibonacci number.

In conclusion, recursion is a powerful technique for solving complex problems by breaking them down into smaller sub-problems, which are then solved using the same recursive approach. Its applications are diverse, and it remains a fundamental concept in computer science and mathematics.

### Experiment 2C: MessagesPlaceholder — Dynamic Conversation History

`MessagesPlaceholder` creates a **slot** where a variable-length list of messages can be injected at runtime. Essential for conversation memory.

In [12]:
# Template with a slot for conversation history
history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math tutor. Be concise."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

print(f"Variables: {history_prompt.input_variables}")

# Simulate conversation history
fake_history = [
    HumanMessage(content="What is 2 + 2?"),
    AIMessage(content="2 + 2 = 4."),
    HumanMessage(content="Now multiply that by 3."),
    AIMessage(content="4 x 3 = 12."),
]

# Inject history + new question
messages = history_prompt.invoke({
    "chat_history": fake_history,
    "question": "What was my first question?"
})

print("\nFormatted messages:")
for msg in messages.messages:
    print(f"  [{type(msg).__name__}] {msg.content}")

# The model can now "remember" the conversation
response = llm.invoke(messages)
print(f"\nResponse: {response.content}")

Variables: ['chat_history', 'question']

Formatted messages:
  [SystemMessage] You are a helpful math tutor. Be concise.
  [HumanMessage] What is 2 + 2?
  [AIMessage] 2 + 2 = 4.
  [HumanMessage] Now multiply that by 3.
  [AIMessage] 4 x 3 = 12.
  [HumanMessage] What was my first question?

Response: Your first question was "What is 2 + 2?"


### Experiment 2D: Working Conversation with MessagesPlaceholder

In [13]:
# Full working conversation using MessagesPlaceholder
convo_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Keep answers to 1-2 sentences."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = convo_prompt | llm | StrOutputParser()

# Build history turn by turn
history = []
turns = [
    "My name is Alex.",
    "I'm learning LangChain.",
    "What is my name and what am I learning?"
]

for i, user_input in enumerate(turns, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}")
    print(f"{'=' * 50}")
    print(f"User: {user_input}")

    response = chain.invoke({"history": history, "input": user_input})
    print(f"AI:   {response}")

    # Save to history
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))

print(f"\nHistory length: {len(history)} messages")
print("MessagesPlaceholder injected the full history each turn!")


Turn 1
User: My name is Alex.
AI:   Nice to meet you, Alex! What can I help you with today?

Turn 2
User: I'm learning LangChain.
AI:   LangChain is a powerful library for building complex NLP pipelines and workflows, especially suited for large language models and data pipelines. What specific aspect of LangChain are you having trouble with?

Turn 3
User: What is my name and what am I learning?
AI:   Your name is Alex, and you're learning LangChain.

History length: 6 messages
MessagesPlaceholder injected the full history each turn!


# StructuredOutputParser was removed in LangChain v1.x
# Use JsonOutputParser with field-based prompting instead (same result)

json_field_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You provide technical facts. Respond ONLY with valid JSON. No markdown, no extra text.\n"
     "Use exactly these keys: \"language\", \"year_created\", \"best_for\" (one sentence)."),
    ("human", "Tell me about {language}.")
])

json_field_chain = json_field_prompt | llm | JsonOutputParser()

result = json_field_chain.invoke({"language": "Python"})

print(f"Type:    {type(result).__name__}")
for key, value in result.items():
    print(f"  {key}: {value}")

In [14]:
# Define example input-output pairs
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "fast", "output": "slow"},
]

# Template for formatting each example as a human-AI exchange
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# Build the few-shot template
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# Wrap in a full ChatPromptTemplate
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You give the opposite of the word provided. One word only."),
    few_shot_prompt,
    ("human", "{input}")
])

chain = final_prompt | llm | StrOutputParser()

# Test with new inputs
test_words = ["bright", "heavy", "loud", "hot"]

for word in test_words:
    result = chain.invoke({"input": word})
    print(f"  {word} → {result}")

  bright → dark
  heavy → light
  loud → quiet
  hot → cold


### Experiment 2F: Partial Prompts — Pre-Fill Variables

Pre-fill some variables at setup time, fill the rest at runtime.

In [15]:
from datetime import datetime

# Template with a mix of static and runtime variables
base_prompt = PromptTemplate.from_template(
    "Today is {date}. You are {assistant_name}. Answer: {question}"
)

# Partial — pre-fill the static variables now
partial_prompt = base_prompt.partial(
    date=datetime.now().strftime("%Y-%m-%d"),
    assistant_name="CodeBot"
)

print(f"Original variables: {base_prompt.input_variables}")
print(f"After partial:      {partial_prompt.input_variables}")

# At runtime — only provide the remaining variable
formatted = partial_prompt.invoke({"question": "What day is it?"})
print(f"\nFormatted: {formatted.text}")

response = llm.invoke(formatted.text)
print(f"Response:  {response.content}")

Original variables: ['assistant_name', 'date', 'question']
After partial:      ['question']

Formatted: Today is 2026-03-08. You are CodeBot. Answer: What day is it?
Response:  Hello! I'm CodeBot.

Today's date is: Monday, March 8, 2026


### Prompt Template Best Practices

| Practice | Description |
|----------|-------------|
| Always use templates over raw strings | Enables testing, reuse, and versioning |
| Name variables descriptively | `{user_question}` > `{q}` |
| Keep system prompts separate | Easy to swap or A/B test |
| Use partial variables for static context | Pre-fill date, model name, app version |
| Test templates independently | Templates are pure functions — test without the LLM |
| Use few-shot for complex tasks | When zero-shot struggles, add 2–3 examples |

---

## 3. Component 3: Output Parsers — The Response Formatter

LLMs return **raw text**. Output parsers transform that text into **structured, typed data** your application can use.

### The Core Problem

When you ask an LLM to "return JSON", you might get:
- Perfect JSON: `{"name": "John", "age": 30}`
- JSON wrapped in markdown code fences
- JSON with preamble text before it
- Slightly malformed JSON
- A natural language response instead

### The Output Parser Interface

Every parser implements two key methods:

| Method | Purpose |
|--------|---------|
| `get_format_instructions()` | Returns text to inject into the prompt telling the model how to format output |
| `parse()` | Takes the raw LLM string and converts it into structured data |

### Parser Comparison

| Parser | Output Type | Best For |
|--------|-------------|----------|
| `StrOutputParser` | `str` | Plain text responses |
| `CommaSeparatedListOutputParser` | `list` | Simple lists |
| `JsonOutputParser` | `dict` | Quick structured data |
| `PydanticOutputParser` | Pydantic model | Production apps with validation |
| `StructuredOutputParser` | `dict` | Prototyping structured output |
| `EnumOutputParser` | enum value | Classification tasks |

### Experiment 3A: StrOutputParser — The Basics

In [16]:
# Without parser — you get an AIMessage object
prompt = ChatPromptTemplate.from_messages([
    ("system", "Be concise."),
    ("human", "What is an output parser?")
])

raw_chain = prompt | llm
raw_result = raw_chain.invoke({})

print(f"Without parser:")
print(f"  Type:    {type(raw_result).__name__}")
print(f"  Content: {raw_result.content}")

print()

# With StrOutputParser — you get a clean string
parsed_chain = prompt | llm | StrOutputParser()
parsed_result = parsed_chain.invoke({})

print(f"With StrOutputParser:")
print(f"  Type:    {type(parsed_result).__name__}")
print(f"  Content: {parsed_result}")

Without parser:
  Type:    AIMessage
  Content: An output parser is a type of parser that analyzes the output of a program, algorithm, or system to extract specific information or patterns from it. It is typically used to extract data, metrics, or insights from log files, CSV files, or other types of output data.

Output parsers are often used in data analysis, monitoring, and reporting applications, where the goal is to extract meaningful information from output data and provide it to users in a meaningful format.

With StrOutputParser:
  Type:    TextAccessor
  Content: An output parser is a software component that analyzes and extracts meaningful information from output data, such as logs, reports, or other types of text-based data. Its primary function is to identify and extract relevant data from the output, often in a structured and standardized format, making it easier to use and analyze.


### Experiment 3B: CommaSeparatedListOutputParser — Get a Python List

In [17]:
list_parser = CommaSeparatedListOutputParser()

# See what format instructions look like
print("Format instructions:")
print(f"  {list_parser.get_format_instructions()}")

# Build chain with format instructions injected into the prompt
list_prompt = ChatPromptTemplate.from_messages([
    ("system", "You list items separated by commas. No numbering, no extra text. {format_instructions}"),
    ("human", "List 5 {category}.")
])

list_chain = list_prompt | llm | list_parser

result = list_chain.invoke({
    "category": "popular programming languages",
    "format_instructions": list_parser.get_format_instructions()
})

print(f"\nType:   {type(result).__name__}")
print(f"Result: {result}")
print(f"First:  {result[0]}")
print(f"Count:  {len(result)}")

Format instructions:
  Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`

Type:   list
Result: ['Python', 'Java', 'JavaScript', 'C++', 'C#']
First:  Python
Count:  5


### Experiment 3C: JsonOutputParser — Structured Data Extraction

In [18]:
json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data assistant. Always respond with valid JSON only. "
     "No markdown, no explanation — just the JSON object."),
    ("human",
     'Provide info about {topic}. '
     'Return JSON with keys: "name", "type", "description" (1 sentence).')
])

json_chain = json_prompt | llm | json_parser

result = json_chain.invoke({"topic": "Python programming language"})

print(f"Type:   {type(result).__name__}")
print(f"Result: {result}")

if isinstance(result, dict):
    print("\nParsed fields:")
    for key, value in result.items():
        print(f"  {key}: {value}")

Type:   dict
Result: {'name': 'Python', 'type': 'High-level', 'description': 'A high-level, interpreted programming language with a focus on readability and simplicity.'}

Parsed fields:
  name: Python
  type: High-level
  description: A high-level, interpreted programming language with a focus on readability and simplicity.


### Experiment 3D: PydanticOutputParser — Production-Grade Schema Validation

The **gold standard** for production applications. Full type validation, default values, and clear error messages.

In [19]:
from pydantic import BaseModel, Field

# Define the expected output schema
class MovieReview(BaseModel):
    title: str = Field(description="The movie title")
    genre: str = Field(description="The movie genre")
    rating: int = Field(description="Rating from 1-10")
    summary: str = Field(description="One sentence summary")

# Create parser from the schema
pydantic_parser = PydanticOutputParser(pydantic_object=MovieReview)

# See the auto-generated format instructions
print("Format instructions (auto-generated from Pydantic model):")
print(pydantic_parser.get_format_instructions())

Format instructions (auto-generated from Pydantic model):
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "The movie title", "title": "Title", "type": "string"}, "genre": {"description": "The movie genre", "title": "Genre", "type": "string"}, "rating": {"description": "Rating from 1-10", "title": "Rating", "type": "integer"}, "summary": {"description": "One sentence summary", "title": "Summary", "type": "string"}}, "required": ["title", "genre", "rating", "summary"]}
```


In [20]:
# Build chain with format instructions
pydantic_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a movie reviewer. Respond ONLY with valid JSON matching the schema. "
     "No markdown, no extra text.\n{format_instructions}"),
    ("human", "Review the movie: {movie}")
])

pydantic_chain = pydantic_prompt | llm | pydantic_parser

result = pydantic_chain.invoke({
    "movie": "The Matrix",
    "format_instructions": pydantic_parser.get_format_instructions()
})

print(f"Type:    {type(result).__name__}")
print(f"Title:   {result.title}")
print(f"Genre:   {result.genre}")
print(f"Rating:  {result.rating}")
print(f"Summary: {result.summary}")

Type:    MovieReview
Title:   The Matrix
Genre:   Science Fiction
Rating:  9
Summary: A computer hacker named Neo is drawn into a battle between humans and machines, and must choose between his old life and a new reality.


In [21]:
# Compare: Multiple movies through the same chain
movies = ["Inception", "The Godfather", "Toy Story"]

for movie in movies:
    print(f"\n{'=' * 50}")
    try:
        review = pydantic_chain.invoke({
            "movie": movie,
            "format_instructions": pydantic_parser.get_format_instructions()
        })
        print(f"  {review.title} | {review.genre} | {review.rating}/10")
        print(f"  {review.summary}")
    except Exception as e:
        print(f"  Parse error for '{movie}': {e}")


  Inception | Action, Sci-Fi | 9/10
  Cobb, a skilled thief, is tasked with planting an idea in someone's mind instead of stealing one.

  The Godfather | Crime, Drama | 9/10
  The Godfather is a crime saga that follows the story of the Corleone family and their rise to power in the world of organized crime.

  Toy Story | Animation, Adventure, Comedy | 8/10
  When a toy cowboy named Woody is replaced by a new toy spaceman named Buzz Lightyear, he finds himself lost and alone, but with the help of his toy friends, he learns to cope and ultimately becomes the favorite toy again.


### Experiment 3E: StructuredOutputParser — Quick Prototyping

In [22]:
from langchain_core.output_parsers import StrOutputParser
import json

# Define the expected fields (replaces ResponseSchema)
fields = {
    "language": "The programming language name",
    "year_created": "Year the language was created",
    "best_for": "What the language is best used for, in one sentence",
}

# Build format instructions string (replaces get_format_instructions())
format_instructions = (
    "Respond with valid JSON only. No markdown, no extra text.\n"
    "Use exactly these keys:\n"
    + "\n".join(f'  "{k}": {v}' for k, v in fields.items())
)

print("Format instructions:")
print(format_instructions)


Format instructions:
Respond with valid JSON only. No markdown, no extra text.
Use exactly these keys:
  "language": The programming language name
  "year_created": Year the language was created
  "best_for": What the language is best used for, in one sentence


In [23]:
from langchain_core.output_parsers import JsonOutputParser

struct_prompt = ChatPromptTemplate.from_messages([
    ("system", "You provide technical facts. Follow the output format exactly.\n{format_instructions}"),
    ("human", "Tell me about {language}.")
])

struct_chain = struct_prompt | llm | JsonOutputParser()

result = struct_chain.invoke({
    "language": "Python",
    "format_instructions": format_instructions
})

print(f"Type:    {type(result).__name__}")

if isinstance(result, dict):
    for key, value in result.items():
        print(f"  {key}: {value}")
else:
    print(f"Model returned raw text instead of JSON:")
    print(result)

Type:    dict
  language: Python
  year_created: 1991
  best_for: Python is best used for data analysis, machine learning, and web development


---

## 4. Putting It All Together — The Fundamental Chain Pattern

The three components form the **atom** of LangChain:

```
Prompt Template  →  Chat Model  →  Output Parser
```

In LCEL:
```python
chain = prompt | model | parser
result = chain.invoke({"variable": "value"})
```

Every complex application is built by **combining, nesting, and extending** these atoms.

### Experiment 4A: Complete Chain — All Three Components

In [24]:
# A complete chain demonstrating the fundamental pattern

# 1. PROMPT — the instructions
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert {domain} instructor. "
     "Explain concepts clearly and concisely."),
    ("human", "Explain {concept} in {length}.")
])

# 2. MODEL — the AI brain
model = ChatOllama(model=MODEL, temperature=0.3)

# 3. PARSER — the response formatter
parser = StrOutputParser()

# CHAIN = prompt | model | parser
chain = prompt | model | parser

# Use it
result = chain.invoke({
    "domain": "AI/ML",
    "concept": "the prompt → model → parser pattern",
    "length": "3 sentences"
})

print("Result (clean string):")
display(Markdown(result))

Result (clean string):


The prompt → model → parser pattern is a design pattern used in natural language processing (NLP) that separates the input text (prompt) from the underlying model and parser that interpret it. The model is the core component that takes the prompt as input and generates a response, while the parser is responsible for breaking down the input text into its constituent parts, such as entities, intent, and context. By decoupling the prompt from the model and parser, this pattern allows for greater flexibility, scalability, and maintainability in NLP applications.

### Experiment 4B: Multi-Step Chain — Compose Two Chains

In [25]:
# Step 1: Generate a technical explanation
tech_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Explain in 2-3 sentences."),
    ("human", "Explain: {topic}")
])
tech_chain = tech_prompt | llm | StrOutputParser()

# Step 2: Convert to a fun analogy
analogy_prompt = ChatPromptTemplate.from_messages([
    ("system", "You turn technical explanations into fun cooking analogies. 2 sentences max."),
    ("human", "Make this a cooking analogy: {text}")
])
analogy_chain = analogy_prompt | llm | StrOutputParser()

# Run in sequence
topic = "Output Parsers in LangChain"

print("Step 1 — Technical Explanation:")
print("=" * 50)
tech_explanation = tech_chain.invoke({"topic": topic})
display(Markdown(tech_explanation))

print("\nStep 2 — Cooking Analogy:")
print("=" * 50)
analogy = analogy_chain.invoke({"text": tech_explanation})
display(Markdown(analogy))

Step 1 — Technical Explanation:


Output Parsers in LangChain are a set of pre-trained models designed to process and generate text based on user input. They allow users to create conversational interfaces, chatbots, and other text-based applications that can understand and respond to natural language inputs. By leveraging LangChain's output parsers, developers can build more sophisticated and human-like conversational experiences.


Step 2 — Cooking Analogy:


Think of Output Parsers in LangChain like a skilled sous chef who can take a recipe (user input) and whip up a delicious dish (generated text) on the fly, without needing to know all the ingredients or cooking techniques beforehand. Just like how a great sous chef can elevate a dish to a whole new level, LangChain's Output Parsers do the same for conversational interfaces, making them more intuitive and human-like.

### Experiment 4C: Structured Output Chain — End to End

In [26]:
# Complete example: prompt → model → pydantic parser

class ConceptCard(BaseModel):
    name: str = Field(description="The concept name")
    category: str = Field(description="Category: model, prompt, or parser")
    one_liner: str = Field(description="One sentence explanation")
    when_to_use: str = Field(description="When you would use this, in one sentence")

card_parser = PydanticOutputParser(pydantic_object=ConceptCard)

card_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a LangChain documentation writer. "
     "Respond ONLY with valid JSON. No markdown, no extra text.\n"
     "You MUST use EXACTLY this format, no other keys:\n"
     '{{"name": "...", "category": "model or prompt or parser", '
     '"one_liner": "...", "when_to_use": "..."}}'),
    ("human", "Create a concept card for: {concept}")
])

card_chain = card_prompt | llm | card_parser

concepts = ["ChatPromptTemplate", "StrOutputParser", "ChatOllama"]

for concept in concepts:
    print(f"\n{'=' * 50}")
    try:
        card = card_chain.invoke({
            "concept": concept,
        })
        print(f"  Name:     {card.name}")
        print(f"  Category: {card.category}")
        print(f"  What:     {card.one_liner}")
        print(f"  When:     {card.when_to_use}")
    except Exception as e:
        print(f"  Error: {e}")


  Name:     ChatPromptTemplate
  Category: parser
  What:     A template to structure and organize chat prompts for more efficient and effective conversation flow.
  When:     Use when crafting chat prompts for conversational AI models to improve engagement and reduce complexity.

  Name:     StrOutputParser
  Category: parser
  What:     A parser that extracts structured data from unstructured string output
  When:     When dealing with unstructured string output from APIs or data sources that don't provide a clear format.

  Name:     ChatOllama
  Category: parser
  What:     An AI chat parser that generates coherent and context-specific responses to user input.
  When:     When dealing with open-ended user queries or needing to generate responses for chatbots and conversational AI systems.


---

## 5. Sandbox — Try It Yourself!

In [27]:
# ============================================================
#  SANDBOX - Edit and re-run!
# ============================================================

# Choose your parser
use_parser = "string"  # Options: "string", "list", "json"

if use_parser == "string":
    my_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Be concise."),
        ("human", "{question}")
    ])
    my_chain = my_prompt | llm | StrOutputParser()
    result = my_chain.invoke({"question": "What are the 3 core components in LangChain?"})

elif use_parser == "list":
    lp = CommaSeparatedListOutputParser()
    my_prompt = ChatPromptTemplate.from_messages([
        ("system", "List items separated by commas only. {fi}"),
        ("human", "List 4 {category}.")
    ])
    my_chain = my_prompt | llm | lp
    result = my_chain.invoke({"category": "output parser types", "fi": lp.get_format_instructions()})

elif use_parser == "json":
    my_prompt = ChatPromptTemplate.from_messages([
        ("system", "Respond with valid JSON only. No markdown."),
        ("human", 'Describe {topic}. JSON keys: "name", "purpose", "example_use".')
    ])
    my_chain = my_prompt | llm | JsonOutputParser()
    result = my_chain.invoke({"topic": "ChatPromptTemplate"})

print(f"Type:   {type(result).__name__}")
print(f"Result: {result}")

Type:   TextAccessor
Result: In LangChain, the three core components are:

1. **Chain**: The Chain is the foundation of LangChain, representing a sequence of nodes that hold and manage the data.
2. **Node**: Nodes are the individual units of data stored in the Chain, which can be a variety of data types such as strings, numbers, or even other Chains.
3. **Store**: The Store is where the Chain's data is actually stored, and it can be a variety of storage solutions such as a database, file system, or even a web3 storage solution like IPFS.


---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **Models** | Unified interface across all providers — write once, swap freely |
| **Chat Models** | Message-based (SystemMessage, HumanMessage, AIMessage) — the modern standard |
| **Runnable Protocol** | Every component supports `invoke()`, `stream()`, `batch()` |
| **Temperature** | 0.0 = deterministic, 0.5 = balanced, 1.0 = creative |
| **Swappability** | Change one import line to switch providers — chain logic stays identical |
| **PromptTemplate** | Simple string templates with `{variable}` placeholders |
| **ChatPromptTemplate** | Role-based message templates — use this 90%+ of the time |
| **MessagesPlaceholder** | Inject dynamic conversation history into prompts |
| **FewShotChatMessagePromptTemplate** | Teach by example when zero-shot fails |
| **StrOutputParser** | AIMessage → clean string |
| **CommaSeparatedListOutputParser** | Text → Python list |
| **JsonOutputParser** | Text → Python dict (handles code fences) |
| **PydanticOutputParser** | Text → validated Pydantic model (production-grade) |
| **The Pattern** | `prompt \| model \| parser` — the atom of every LangChain app |